<a href="https://colab.research.google.com/github/zhabibi-z/house_price_prediction/blob/main/HousePricePrediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# House Price Prediction — Ames (Kaggle *House Prices*)

A single, coherent pipeline: **Phase 1** sanitization → **Phase 2** feature engineering →
**Phase 3** EDA → **Phase 4** AutoGluon modeling → one submission. The target is
`log1p(SalePrice)` and the metric is RMSE on that log target (RMSLE).


In [ ]:
# One-time install (Colab). Locally: pip install -r requirements.txt
# !pip install -q autogluon.tabular[extra]==1.2
import numpy as np
import pandas as pd
import scipy.stats as stats
from autogluon.tabular import TabularPredictor

SEED = 42
np.random.seed(SEED)
ID_COL, TARGET_COL, LOG_TARGET_COL = 'Id', 'SalePrice', 'log_SalePrice'


## Data

The notebook expects the Kaggle competition files `train.csv` and `test.csv` (with `Id` as
the index). In Colab, set up your Kaggle API token and run the download cell; if the files
are already present (e.g. run locally) the download is skipped.


In [ ]:
import os
if not (os.path.exists('train.csv') and os.path.exists('test.csv')):
    # Kaggle download (requires ~/.kaggle/kaggle.json)
    os.system('pip install -q kaggle')
    os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
    if os.path.exists('kaggle.json'):
        os.system('cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json')
    os.system('kaggle competitions download -c house-prices-advanced-regression-techniques')
    os.system('unzip -o house-prices-advanced-regression-techniques.zip -d .')

train_df = pd.read_csv('train.csv', index_col=ID_COL)
test_df = pd.read_csv('test.csv', index_col=ID_COL)
print('train:', train_df.shape, ' test:', test_df.shape)


## PHASE 1 — Sanitization & dirty-value correction

Extract and log-transform the target, then a single `preprocess_phase1` handles structural
NAs, numeric missingness, and known dirty values consistently for train and test. Outliers
are removed from the training data only.


In [ ]:
y_train = np.log1p(train_df[TARGET_COL])
X_train_raw = train_df.drop(columns=[TARGET_COL])

def preprocess_phase1(df, is_train=False, train_lotfrontage_median=None):
    df = df.copy()
    structural_na = ['PoolQC','MiscFeature','Alley','Fence','FireplaceQu','GarageType',
                     'GarageFinish','GarageQual','GarageCond','BsmtQual','BsmtCond',
                     'BsmtExposure','BsmtFinType1','BsmtFinType2']
    for c in structural_na:
        if c in df: df[c] = df[c].fillna('None')
    if 'LotFrontage' in df:
        if is_train:
            df['LotFrontage'] = df.groupby('Neighborhood')['LotFrontage'].transform(
                lambda s: s.fillna(s.median()))
            train_lotfrontage_median = df['LotFrontage'].median()
        df['LotFrontage'] = df['LotFrontage'].fillna(train_lotfrontage_median)
    for c in ['GarageYrBlt','GarageArea','GarageCars','MasVnrArea','BsmtFinSF1','BsmtFinSF2',
              'BsmtFullBath','BsmtHalfBath','BsmtUnfSF','TotalBsmtSF']:
        if c in df: df[c] = df[c].fillna(0)
    if 'MasVnrType' in df: df['MasVnrType'] = df['MasVnrType'].fillna('None')
    if 'Electrical' in df: df['Electrical'] = df['Electrical'].fillna(df['Electrical'].mode()[0])
    if 'GarageYrBlt' in df: df.loc[df['GarageYrBlt'] == 2207, 'GarageYrBlt'] = df['YearBuilt']
    if 'Utilities' in df: df = df.drop(columns=['Utilities'])
    return df, train_lotfrontage_median

X_train, lf_median = preprocess_phase1(X_train_raw, is_train=True)
X_test, _ = preprocess_phase1(test_df, is_train=False, train_lotfrontage_median=lf_median)

# Outlier removal (train only): large living area sold cheap (community heuristic)
mask = (X_train['GrLivArea'] > 4000) & (np.expm1(y_train) < 300000)
X_train, y_train = X_train[~mask], y_train[~mask]
print(f'Removed {int(mask.sum())} outliers; train now {X_train.shape}')


## PHASE 2 — Feature Engineering

Age features, total-area/bath aggregates, polynomial `OverallQual`, and quality×area
interactions; then log-transform skewed numerics. (A target-encoded neighborhood index was
intentionally dropped — it leaks the target into the features; `Neighborhood` stays
categorical for AutoGluon to encode.)


In [ ]:
def engineer(df):
    df = df.copy()
    # Age features
    df['Age'] = (df['YrSold'] - df['YearBuilt']).clip(lower=0)
    df['RemodelAge'] = (df['YrSold'] - df['YearRemodAdd']).clip(lower=0)
    df['GarageAge'] = (df['YrSold'] - df['GarageYrBlt']).clip(lower=0)
    df.loc[df['GarageYrBlt'] == 0, 'GarageAge'] = 0
    df = df.drop(columns=['YearBuilt','YearRemodAdd','YrSold','MoSold','GarageYrBlt'])
    # Total-area / bath / porch aggregates
    df['TotalSF'] = df['1stFlrSF'] + df['2ndFlrSF'] + df['TotalBsmtSF']
    df['TotalBath'] = df['FullBath'] + 0.5*df['HalfBath'] + df['BsmtFullBath'] + 0.5*df['BsmtHalfBath']
    df['TotalPorchSF'] = df['OpenPorchSF'] + df['EnclosedPorch'] + df['3SsnPorch'] + df['ScreenPorch']
    # Polynomial quality
    df['OverallQual_sq'] = df['OverallQual']**2
    df['OverallQual_cub'] = df['OverallQual']**3
    # Quality x area interactions (no target leakage)
    df['TotalSF_x_Qual'] = df['TotalSF'] * df['OverallQual']
    df['GrLivArea_x_Qual'] = df['GrLivArea'] * df['OverallQual']
    df['HasPool'] = (df['PoolArea'] > 0).astype(int)
    return df

X_train, X_test = engineer(X_train), engineer(X_test)

def log_skewed(df, cols, threshold=0.75):
    for c in cols:
        if (df[c] > 0).any() and df[c].nunique() > 1 and stats.skew(df[c].dropna()) > threshold:
            df[c] = np.log1p(df[c].clip(lower=0))
    return df

skew_cols = X_train.select_dtypes(include=np.number).columns
X_train = log_skewed(X_train, skew_cols)
X_test = log_skewed(X_test, skew_cols)
print('features:', X_train.shape[1], ' remaining train NaNs:', int(X_train.isnull().sum().sum()))


In [ ]:
# Assemble modeling frames used by EDA and AutoGluon
train_data = X_train.copy()
train_data[LOG_TARGET_COL] = y_train
test_data = X_test.copy()
print('train_data:', train_data.shape, ' test_data:', test_data.shape)


## PHASE 3 — Exploratory Data Analysis

EDA precedes modeling. The plots use the engineered `train_data`.


In [ ]:
import matplotlib.pyplot as plt
try:
    import seaborn as sns
    _has_sns = True
except ImportError:
    _has_sns = False

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].hist(np.expm1(train_data[LOG_TARGET_COL]), bins=40, color='skyblue')
ax[0].set_title('SalePrice (original, right-skewed)'); ax[0].set_xlabel('SalePrice')
ax[1].hist(train_data[LOG_TARGET_COL], bins=40, color='lightcoral')
ax[1].set_title('log1p(SalePrice) — near-normal'); ax[1].set_xlabel('log(1+SalePrice)')
plt.tight_layout(); plt.show()


In [ ]:
# Top correlations with the (log) target
num = train_data.select_dtypes(include=np.number)
corr = num.corr()[LOG_TARGET_COL].drop(LOG_TARGET_COL).sort_values(ascending=False)
print('Top 10 positively correlated features with log SalePrice:')
print(corr.head(10).round(3))


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].scatter(train_data['GrLivArea'], train_data[LOG_TARGET_COL], s=8, alpha=0.4)
ax[0].set_title('GrLivArea vs log SalePrice'); ax[0].set_xlabel('GrLivArea'); ax[0].set_ylabel('log(1+SalePrice)')
train_data.boxplot(column=LOG_TARGET_COL, by='OverallQual', ax=ax[1])
ax[1].set_title('OverallQual vs log SalePrice'); plt.suptitle(''); plt.tight_layout(); plt.show()


## PHASE 4 — Modeling (AutoGluon)

A single AutoGluon `TabularPredictor` on the engineered features, evaluated with RMSLE
(RMSE on the log target), then one submission. AutoGluon performs its own internal
validation and ensembling.


In [ ]:
predictor = TabularPredictor(
    label=LOG_TARGET_COL,
    problem_type='regression',
    eval_metric='root_mean_squared_error',  # RMSE on log target == RMSLE
    path='AutogluonModels',
).fit(train_data, presets='best_quality', time_limit=3600)


In [ ]:
leaderboard = predictor.leaderboard(silent=True)
display(leaderboard[['model', 'score_val', 'fit_time']].head(10))
print('Best validation RMSLE: %.4f' % -leaderboard['score_val'].iloc[0])


In [ ]:
# One submission: inverse-transform predictions back to SalePrice
preds = np.expm1(predictor.predict(test_data))
submission = pd.DataFrame({ID_COL: test_data.index, TARGET_COL: preds})
submission.to_csv('submission.csv', index=False)
print('Wrote submission.csv'); display(submission.head())
